# Solar PPA â€” AndalucÃ­a (Pieza 5)

Phase-2 / Pieza 5 of the mibel-derivatives module.

Reference asset: a **100 MW** utility-scale PV plant in AndalucÃ­a
(lat 37.4, lon -5.0, the PVGIS reference site near Sevilla). Hourly
generation is built from the curated PVGIS panel
(`data/curated/pvgis_panel.parquet`, SARAH3, 2019-2023).

Contract (v1): **80% fixed + 20% spot**, no caps / floors. The fixed
fraction is sold at a fixed `strike`; the rest settles at the hourly
MIBEL spot.

The notebook covers:
1. The solar resource (PVGIS): diurnal & seasonal shape, capacity factor.
2. Capture price â€” historical (OMIE x PVGIS) vs simulated.
3. PPA valuation over 10 years at `N_PATHS=1000`.
4. Sensitivities: strike, fixed/spot split, renewable-penetration
   (cannibalisation) scenarios.

Pricer: `mibel_derivatives.products.ppa`. The valuation arithmetic is
validated in `tests/products/test_ppa.py`.

In [ ]:
from __future__ import annotations

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from mibel_derivatives.data import _paths
from mibel_derivatives.products.ppa import (
    PPA,
    PriceModel,
    price_ppa,
    representative_year_profile,
    simulate_price_paths,
    simulate_production_paths,
)

plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["axes.grid"] = True
CAPACITY_MW = 100.0

## 1. Solar resource analysis (PVGIS)

The curated panel holds two stock geometries for the site: fixed-tilt
35deg south (`fixed_35deg_south`) and single-axis horizontal N-S
(`one_axis_horizontal_ns`). PV power `p_w` is for a 1 kWp reference, so
the hourly capacity factor is `p_w / 1000`.

In [ ]:
panel = pd.read_parquet(_paths.curated_path("pvgis_panel.parquet"))
panel["cf"] = (panel["p_w"] / 1000.0).clip(0.0, 1.0)
panel["hour"] = panel["dt_utc"].dt.hour
panel["month"] = panel["dt_utc"].dt.month

summary = (
    panel.groupby("config")["cf"]
    .agg(mean_cf="mean", max_cf="max")
    .assign(equiv_full_load_hours=lambda d: d["mean_cf"] * 8760)
)
summary

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
for cfg, g in panel.groupby("config"):
    g.groupby("hour")["cf"].mean().plot(ax=ax1, marker="o", label=cfg)
    g.groupby("month")["cf"].mean().plot(ax=ax2, marker="o", label=cfg)
ax1.set(title="Mean capacity factor by hour (UTC)", xlabel="hour", ylabel="CF")
ax2.set(title="Mean capacity factor by month", xlabel="month", ylabel="CF")
ax1.legend(); ax2.legend()
plt.tight_layout()

In [ ]:
# Typical-year 8760-hour profiles used by the pricer.
prof_fixed = representative_year_profile("fixed_35deg_south")
prof_axis = representative_year_profile("one_axis_horizontal_ns")
print("fixed-tilt : hours", prof_fixed.size,
      "mean CF", round(float(prof_fixed.mean()), 4),
      "-> EFLH", round(float(prof_fixed.mean()) * 8760))
print("one-axis   : hours", prof_axis.size,
      "mean CF", round(float(prof_axis.mean()), 4),
      "-> EFLH", round(float(prof_axis.mean()) * 8760))

## 2. Capture price â€” historical vs simulated

The **capture price** is the generation-weighted average spot price the
plant realises:

$$\text{capture} = \frac{\sum_h g_h\, S_h}{\sum_h g_h}
\quad\le\quad \overline{S} = \frac{1}{H}\sum_h S_h = \text{baseload}.$$

Solar output peaks at midday when the price is depressed (the duck
curve), so capture < baseload. We compute the **historical** capture by
joining PVGIS production with the OMIE day-ahead Spain price, year by
year, then compare to the model's simulated capture.

In [ ]:
omie = pd.read_parquet(_paths.curated_path("omie_spot_es_2019_2024.parquet"))
omie = omie[["datetime_utc", "price_eur_mwh"]].rename(
    columns={"datetime_utc": "dt_utc", "price_eur_mwh": "price"}
)

pv = panel[panel["config"] == "fixed_35deg_south"][["dt_utc", "cf"]]
hist = pv.merge(omie, on="dt_utc", how="inner")
hist["year"] = hist["dt_utc"].dt.year

rows = []
for y, g in hist.groupby("year"):
    gen = g["cf"].to_numpy()
    px = g["price"].to_numpy()
    capture = float((gen * px).sum() / gen.sum())
    baseload = float(px.mean())
    rows.append({"year": y, "capture": capture, "baseload": baseload,
                 "ratio": capture / baseload})
hist_capture = pd.DataFrame(rows).set_index("year")
hist_capture.round(3)

In [ ]:
# Simulated capture from the reduced-form price model, same profile.
model = PriceModel(baseload=float(hist_capture["baseload"].mean()),
                   cannibalisation=0.45, diurnal_amplitude=0.30, sigma=0.25, ar1=0.80)
ppa_ref = PPA(CAPACITY_MW, float(prof_fixed.mean()), 0.80, 0.20, 55.0, 10)
sim = price_ppa(ppa_ref, pv_profile=prof_fixed, price_model=model, n_paths=1000, seed=2024)

print("historical capture ratio (2019-2023): mean",
      round(float(hist_capture["ratio"].mean()), 3),
      "range", round(float(hist_capture["ratio"].min()), 3),
      "-", round(float(hist_capture["ratio"].max()), 3))
print("simulated  capture ratio            :",
      round(sim.capture_ratio, 3),
      f"(capture {sim.capture_price_mean:.2f} vs baseload {sim.baseload_price_mean:.2f})")

In [ ]:
fig, ax = plt.subplots()
hist_capture["ratio"].plot(ax=ax, marker="o", label="historical (OMIE x PVGIS)")
ax.axhline(sim.capture_ratio, color="C1", ls="--", label="simulated (model)")
ax.axhline(1.0, color="grey", ls=":", label="baseload (ratio=1)")
ax.set(title="Solar capture ratio, Andalucia fixed-tilt", xlabel="year",
       ylabel="capture / baseload")
ax.legend()

The capture discount is structural and persistent: solar in southern
Spain captures materially less than baseload, and the gap widens as
renewable penetration grows (more midday cannibalisation). The reduced-form
model's `cannibalisation` knob is the lever for forward scenarios.

## 3. PPA valuation â€” 10 years, N_PATHS = 1000

Per-hour cashflow to the generator:

$$\text{payoff}_h = g_h\,(\text{strike}\cdot f_{\text{fix}}
+ S_h\cdot f_{\text{spot}} - \text{cost}),$$

summed to an annual cashflow and present-valued over 10 years at a flat
7% real discount rate (mid-year annuity).

In [ ]:
ppa = PPA(capacity_mw=CAPACITY_MW, plant_factor=float(prof_fixed.mean()),
          fixed_pct=0.80, spot_pct=0.20, strike=55.0, duration_years=10)
res = price_ppa(ppa, pv_profile=prof_fixed, price_model=model, n_paths=1000, seed=2024)

print(f"annual generation   : {res.annual_generation_mwh:,.0f} MWh")
print(f"capture price        : {res.capture_price_mean:.2f} EUR/MWh "
      f"(baseload {res.baseload_price_mean:.2f}, ratio {res.capture_ratio:.3f})")
print(f"PPA value (10y PV)   : {res.value/1e6:,.2f} M EUR  +/- {res.std_error/1e6:.3f}")
print(f"levelised value      : {res.value_per_mwh:.2f} EUR/MWh")

In [ ]:
fig, ax = plt.subplots()
ax.hist(res.per_path_value / 1e6, bins=40, color="C0", alpha=0.8)
ax.axvline(res.value / 1e6, color="C3", ls="--", label=f"mean {res.value/1e6:.1f} M EUR")
ax.set(title="PPA present-value distribution (1000 paths, 10y)",
       xlabel="PV [M EUR]", ylabel="paths")
ax.legend()

## 4. Sensitivities

### 4.1 Strike (fixed-leg price)

In [ ]:
strikes = np.arange(40, 71, 5.0)
vals = [price_ppa(PPA(CAPACITY_MW, float(prof_fixed.mean()), 0.80, 0.20, float(k), 10),
                  pv_profile=prof_fixed, price_model=model, n_paths=1000, seed=2024).value / 1e6
        for k in strikes]
fig, ax = plt.subplots()
ax.plot(strikes, vals, marker="o")
ax.set(title="PPA value vs fixed strike", xlabel="strike [EUR/MWh]", ylabel="PV [M EUR]")

### 4.2 Fixed / spot split

More fixed allocation transfers price risk to the offtaker and raises the
value to the generator whenever the strike exceeds the capture price.

In [ ]:
splits = [1.0, 0.8, 0.6, 0.4, 0.2, 0.0]
rows = []
for f in splits:
    rr = price_ppa(PPA(CAPACITY_MW, float(prof_fixed.mean()), f, round(1 - f, 4), 55.0, 10),
                   pv_profile=prof_fixed, price_model=model, n_paths=1000, seed=2024)
    rows.append({"fixed_pct": f, "value_MEUR": rr.value / 1e6,
                 "per_MWh": rr.value_per_mwh, "std_MEUR": rr.std_error / 1e6})
split_df = pd.DataFrame(rows)
split_df.round(3)

In [ ]:
fig, ax = plt.subplots()
ax.errorbar(split_df["fixed_pct"], split_df["value_MEUR"],
            yerr=split_df["std_MEUR"], marker="o", capsize=3)
ax.set(title="PPA value vs fixed share", xlabel="fixed_pct", ylabel="PV [M EUR]")

### 4.3 Renewable-penetration scenarios

Higher solar penetration deepens the midday price trough â€” modelled by
raising the `cannibalisation` coefficient. Capture falls and, for a fixed
strike above capture, the fixed leg becomes more valuable insurance.

In [ ]:
betas = [0.0, 0.3, 0.45, 0.6, 0.8]
rows = []
for b in betas:
    m = PriceModel(baseload=model.baseload, cannibalisation=b,
                   diurnal_amplitude=0.30, sigma=0.25, ar1=0.80)
    rr = price_ppa(ppa, pv_profile=prof_fixed, price_model=m, n_paths=1000, seed=2024)
    rows.append({"cannibalisation": b, "capture": rr.capture_price_mean,
                 "capture_ratio": rr.capture_ratio, "value_MEUR": rr.value / 1e6})
pen_df = pd.DataFrame(rows)
pen_df.round(3)

In [ ]:
fig, (axa, axb) = plt.subplots(1, 2, figsize=(13, 4))
axa.plot(pen_df["cannibalisation"], pen_df["capture_ratio"], marker="o")
axa.set(title="Capture ratio vs cannibalisation", xlabel="beta", ylabel="capture / baseload")
axb.plot(pen_df["cannibalisation"], pen_df["value_MEUR"], marker="o", color="C2")
axb.set(title="PPA value vs cannibalisation", xlabel="beta", ylabel="PV [M EUR]")
plt.tight_layout()

## Notes

- The pricer is **injectable**: pass `price_paths` / `production_paths`
  to bypass the internal generators (e.g. for an OMIP-consistent run via
  `simulate_price_paths_from_spot`, which drives `models.spot.simulate`
  (Pieza 2) and the forward level of `models.forward` (Pieza 1)).
- v1 treats each contract year as the same annual scenario times the
  annuity factor; interannual resampling and explicit caps/floors are
  documented extensions in `reports/diagnostics/ppa_solar.md`.
- The reduced-form `cannibalisation` coefficient is calibrated here to
  reproduce the historical capture ratio; a structural penetration model
  is the natural successor.